# 03 · Recomendação no mapa-múndi
    Execute os notebooks 01 e 02 para publicar as observações e as previsões.
    O perfil da base é fixo. O mês seleciona somente os valores terrestres.
    O cluster aparece no mês usado para ajustar o agrupamento. As coordenadas vêm do de-para.

## Configurar caminhos

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
TERRA = ROOT / 'data/terra'
RUN = TERRA / 'analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Escolher o mês, a base e o nicho

In [ ]:
import plotly.express as px

from feynmans_legacy.analog_study import MIN_COVERAGE, profile_criteria, recommend

ANOMES = None  # None usa o ultimo mes observado
CORPO = "lua"
BASE = None  # None usa a primeira base da spec selecionada
NICHO = "Todos"
# Fração mínima do peso do perfil com dados no local. Critérios sem dado saem da nota
# e os pesos restantes são renormalizados. 1.0 exige todos os critérios.
COBERTURA_MINIMA = MIN_COVERAGE


## Buscar os dados do mês

In [3]:
import json

selection = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))
latest = max(selection["anomes_observados"])
ANOMES = latest if ANOMES is None else ANOMES
if ANOMES <= latest:
    origem = "observada"
    values = pd.read_parquet(
        TERRA / "terra.parquet", filters=[("anomes", "==", ANOMES)]
    ).drop(columns="anomes")
    if values.empty:
        raise ValueError("Sem observações para o anomes selecionado.")
else:
    origem = "prevista"
    future = pd.read_parquet(
        RUN / "previsoes.parquet", filters=[("anomes", "==", ANOMES)]
    )
    if future.empty:
        raise ValueError("Sem previsões publicadas para o anomes selecionado.")
    values = future.pivot(  # noqa: PD010
        index="location_id", columns="variavel", values="valor_previsto"
    ).reset_index()
values = values.merge(
    pd.read_parquet(RUN / "estaticas.parquet"),
    on="location_id", how="left", validate="one_to_one",
)

## Aplicar o perfil e ordenar os locais

In [ ]:
reference = pd.read_parquet(ROOT / "data" / CORPO / f"{CORPO}.parquet")
if BASE is None:
    if reference.empty:
        raise ValueError(f"Sem bases cadastradas na spec de {CORPO}.")
    BASE = reference["base_id"].iloc[0]
criteria = profile_criteria(ROOT / "config/nichos.json", CORPO, NICHO, reference, BASE)
ranking = recommend(
    values, pd.read_parquet(RUN / "coordenadas.parquet"), criteria,
    min_coverage=COBERTURA_MINIMA,
)
if ranking.empty:
    raise ValueError("Nenhum local com cobertura mínima e coordenadas para essa consulta.")


## Adicionar os clusters do mês

In [5]:
clusters = pd.read_parquet(RUN / "clusters.parquet").query("anomes == @ANOMES")
ranking = ranking.merge(clusters[["location_id", "cluster"]], on="location_id", how="left",
                        validate="one_to_one")
ranking["recomendado"] = ["Recomendado"] + ["Candidato"] * (len(ranking) - 1)


## Conferir os primeiros colocados

In [ ]:
print(f"{origem.capitalize()} | {len(ranking):,} locais elegíveis de {len(values):,} consultados")
completos = ranking["cobertura"].ge(1 - 1e-9).sum()
print(f"{completos:,} com todos os critérios; {len(ranking) - completos:,} com cobertura parcial")
summary_columns = [
    "location_id", "nota", "cobertura", "criterios_disponiveis", "latitude", "longitude", "cluster",
]
print(ranking[summary_columns].head(10).to_string(index=False))

## Exibir os locais no mapa-múndi

In [ ]:
LIMITE_MAPA = 2000
points = ranking.head(LIMITE_MAPA).copy()
points["cluster"] = points["cluster"].astype("string").fillna("Indisponível")
figure = px.scatter_geo(
    points, lat="latitude", lon="longitude", color="recomendado", hover_name="location_id",
    hover_data=[
        "nota", "cobertura", "cluster",
        *criteria, *[f"alvo_{variable}" for variable in criteria],
    ],
    projection="natural earth",
    title=f"{BASE} · {NICHO} · {ANOMES} ({origem}) · {len(points):,} de {len(ranking):,} locais",
)
figure.show()